# 04. Preparación de datos

Usar la muestra existente y distinguir protocolos ejecutados de un procedimiento recomendado sin fuga.

**Dataset principal:** `data/nuevo_dataset/cardio_train.csv`. Este cuaderno no ejecuta entrenamientos. Los resultados históricos se leen de archivos conservados; no se sustituyen por nuevas ejecuciones.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/nuevo_dataset/cardio_train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.experimento import (
    ORIGINAL, RESULTADOS, VARIABLES, NOMBRES, cargar_datos, reglas_calidad,
    cargar_metricas, comparar_ejecuciones, tabla_configuracion, verificar_configuracion,
    construir_cnn, tabla_modelo, dibujar_arquitectura, guardar_figura_nueva,
    importancia_guardada, resumen_dataset, verificar_predicciones, tensorflow_disponible,
)

In [2]:
muestra = cargar_datos(RESULTADOS / "cardio_15000.csv")
limpio = cargar_datos(RESULTADOS / "cardio_limpio.csv")
assert len(muestra) == 15000 and muestra.id.is_unique
assert set(muestra.id).issubset(set(limpio.id))
pd.testing.assert_frame_equal(muestra.set_index("id").sort_index(), limpio.set_index("id").loc[muestra.id].sort_index(), check_dtype=False)
display(muestra.cardio.value_counts().sort_index().to_frame("registros"))
data = muestra.copy()
data["age"] /= 365.25
X, y = data[VARIABLES], data.cardio
print("X:", X.shape, "y:", y.shape)
assert len(VARIABLES) == 11

X: (15000, 11) y: (15000,)


,registros
cardio,
0,7579
1,7421


## Protocolos que produjeron los resultados existentes
La muestra fue obtenida por selección estratificada con semilla 42. **No se vuelve a muestrear.** La ejecución inicial utilizó StandardScaler y particiones explícitas 70/15/15. La definitiva separó 15 % de prueba, ajustó MinMaxScaler sobre el 85 % restante y reservó luego 20 % de ese bloque con `validation_split`. Esto implica 68/17/15 efectivos. La validación influyó en el escalamiento de la ejecución definitiva; la prueba no. No se atribuyen las métricas guardadas al procedimiento recomendado siguiente.

In [3]:
display(comparar_ejecuciones())

,Protocolo,Entrenamiento,Validación,Prueba,Épocas,Parámetros,Accuracy,Precision,Recall,F1,ROC-AUC
0,StandardScaler,10500,2250,2250,44,34529,0.712444,0.716141,0.693621,0.704701,0.775738
1,MinMaxScaler,10200,2550,2250,66,34529,0.731111,0.732601,0.718778,0.725624,0.788458


## Procedimiento recomendado sin fuga: solo preparación, sin entrenamiento
Se separan primero entrenamiento, validación y prueba, con estratificación y semilla fija. El escalador se ajusta exclusivamente al entrenamiento. Este ejemplo usa 70/15/15 y MinMaxScaler; **no es una nueva ejecución evaluada ni reproduce las métricas definitivas**. La auditoría entre particiones compara características sin `id`, ya que un identificador único oculta coincidencias de contenido.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=.50, stratify=y_temp, random_state=42)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
scaler = MinMaxScaler().fit(X_train)
X_train_cnn = scaler.transform(X_train).astype("float32")[..., np.newaxis]
X_val_cnn = scaler.transform(X_val).astype("float32")[..., np.newaxis]
X_test_cnn = scaler.transform(X_test).astype("float32")[..., np.newaxis]
assert scaler.n_samples_seen_ == len(X_train)
for bloque in [X_train_cnn, X_val_cnn, X_test_cnn]:
    assert bloque.shape[1:] == (11, 1)
print("Formas:", X_train_cnn.shape, X_val_cnn.shape, X_test_cnn.shape)
for nombre, a, b in [("train/val", X_train, X_val), ("train/test", X_train, X_test), ("val/test", X_val, X_test)]:
    print(nombre, "patrones predictores compartidos:", len(set(map(tuple, a.to_numpy())) & set(map(tuple, b.to_numpy()))))

Formas: (10500, 11, 1) (2250, 11, 1) (2250, 11, 1)
train/val patrones predictores compartidos: 1
train/test patrones predictores compartidos: 0
val/test patrones predictores compartidos: 0


Si hay patrones coincidentes entre particiones, deben informarse; no implican automáticamente identidad de pacientes. Cambiar a una partición por grupos sería otro protocolo y exigiría resultados nuevos. Las categorías numéricas y el orden fijo de las variables se mantienen para respetar la arquitectura validada.